In [ ]:
%load_ext autoreload
%autoreload 2
    
import numpy as np
import pandas as pd
from glob import glob
import re
from pathlib import Path
import matplotlib.pyplot as plt
from collections import defaultdict


import sys
sys.path.insert(1, '../')
from larfitter import retrieve_responses
from waffles.data_classes.EasyWaveformCreator import EasyWaveformCreator
from utils import PATH_XE_AVERAGES
from utils_deconv import DeconvFitParams 
from utils import setup_response_template
from waffles.np02_utils.load_utils import ch_read_template
from DeconvFitterVDWrapper import DeconvFitterVDWrapper
from utils_monitoring import load_database
from waffles.np02_utils.PlotUtils import matplotlib_plot_WaveformSetGrid
from waffles.np02_utils.AutoMap import ordered_modules_cathode, ordered_modules_membrane, ordered_modules_pmt, getModuleName

# try: 
#     import dunestyle.matplotlib as dunestyle
#     from cycler import cycler
#     plt.rcParams['axes.facecolor'] = '#ffffff'
#     plt.rcParams['figure.facecolor'] = '#ffffff'
#     plt.rcParams['savefig.facecolor'] = '#ffffff'
#     plt.rcParams.update(
#         {
#         "axes.prop_cycle":cycler(color=['#1f77b4',
#                                         '#ff7f0e',
#                                         '#2ca02c',
#                                         '#d62728',
#                                         '#9467bd',
#                                         '#8c564b',
#                                         '#e377c2',
#                                         '#7f7f7f',
#                                         '#bcbd22',
#                                         '#17becf'])
#         })
        
# except Exception as e:
# print("Using mplhep... ", e)
import mplhep
mplhep.style.use(mplhep.style.ROOT)
plt.rcParams.update({
                     'font.size': 14,
                     'grid.linestyle': '--',
                     'axes.grid': True,
                     'figure.autolayout': True,
                     'figure.figsize': [14,6],
                    #  'font.family': 'sans-serif',
                    #  'font.sans-serif': ['Helvetica', 'Helvetica Neue', 'Nimbus Sans', 'Liberation Sans', 'Arial']
                     })

In [ ]:
path_with_analysis = Path(PATH_XE_AVERAGES)
analysis_name = 'average_waveforms_few_cuts-gpiemont'
db = load_database(load_hv = False)
ppm_values = db['ppm'].unique()
print("PPM values:", ppm_values)

In [ ]:
dettypes = ["cathode", "membrane"]

def total_average_waveform(db, path_base: Path, ppm_values):
    from collections import defaultdict
    import numpy as np

    output_all = {}
    chinfo_all = {}
    runs_per_ppm = defaultdict(lambda: defaultdict(list))

    for ppm in ppm_values:
        runs_for_ppm = db[(db['ppm'] == ppm) & (db['status'] == 'stable')]

        for idx, row in runs_for_ppm.iterrows():
            run = row['run']

            for dettype in dettypes:

                if dettype == "cathode":
                    endpoint = 106
                elif dettype == "membrane":
                    endpoint = 107

                matches = list(path_base.glob(f"run*{run}_{dettype}*"))
                if not matches:
                    print(f"Warning: run {run} ({dettype}) not found.")
                    continue

                run_folder = matches[0]

                output_run = {}
                chinfo_run = {}

                retrieve_responses(run_folder, output_run, chinfo_run)

                if endpoint not in output_run:
                    print(f"Warning: endpoint {endpoint} missing in run {run}")
                    continue

                if run not in output_all:
                    output_all[run] = {}
                    chinfo_all[run] = {}

                output_all[run][endpoint] = output_run[endpoint]
                chinfo_all[run][endpoint] = chinfo_run[endpoint]

                runs_per_ppm[ppm][endpoint].append(run)

    for ppm, ep_dict in runs_per_ppm.items():
        mean_output[ppm] = {}

        for ep, run_list in ep_dict.items():
            if not run_list:
                continue

            mean_output[ppm][ep] = {}

            example_run = run_list[0]
            channels = output_all[example_run][ep].keys()

            for ch in channels:
                numerator = 0
                denominator = 0

                for run in run_list:
                    if ch not in output_all[run][ep]:
                        print(f"Missing ch {ch}-{ep} in run {run}")
                        continue

                    wf = output_all[run][ep][ch]
                    counts = chinfo_all[run][ep][ch]['counts']

                    numerator += wf * counts
                    denominator += counts

                if denominator > 0:
                    mean_output[ppm][ep][ch] = numerator / denominator
    return mean_output

In [ ]:
mean_output={}
total_average_waveform(db, path_with_analysis/analysis_name, ppm_values)

In [ ]:
n_ppm = next(iter(mean_output))
dict_ep_channels = { ep: list(mean_output[n_ppm][ep].keys()) for ep in mean_output[n_ppm] }
dict_ep_channels

In [ ]:
wfset = EasyWaveformCreator.create_WaveformSet_dictEndpointCh(dict_endpoint_ch=dict_ep_channels)

In [ ]:
def plot_ratios_all_channels(mean_output, ppm_values):
    ppm_sorted = sorted(ppm_values)
    
    EXCLUDE_MODULES = {"M1(1)", "M1(2)", "M2(2)", "M4(1)"}
    
    labels   = []
    ratio_01 = []
    ratio_1  = []

    for ep in mean_output[ppm_sorted[0]]:
        for ch in mean_output[ppm_sorted[0]][ep]:
            
            if getModuleName(ep, ch) in EXCLUDE_MODULES:
                continue
            
            max_n_ppm = []
            for ppm in ppm_sorted:
                if ppm not in mean_output or ep not in mean_output[ppm] or ch not in mean_output[ppm][ep]:
                    max_n_ppm.append(None)
                else:
                    max_n_ppm.append(np.max(mean_output[ppm][ep][ch]))
            
            if max_n_ppm[0] is None:
                continue 

            labels.append(f"{getModuleName(ep, ch)}: {ep}-{ch}")
            ratio_01.append(max_n_ppm[0] / max_n_ppm[1] if max_n_ppm[1] is not None else None)
            ratio_1.append(max_n_ppm[0]  / max_n_ppm[2] if max_n_ppm[2] is not None else None)

    x = np.arange(len(labels))
    
    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.6), 5))
    
    ax.plot(x, ratio_01, marker='o', label="0 ppm / 0.01 ppm")
    ax.plot(x, ratio_1,  marker='o', label="0 ppm / 1 ppm")

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=12)
    ax.set_xlabel("Modules")
    ax.set_ylabel("Ratio")
    ax.set_title("Average WFs max amplitude ratio")
    ax.legend(fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("Average_max_ratio")
    plt.show()

In [ ]:
plot_ratios_all_channels(mean_output, ppm_values)

In [ ]:
fig, axs = matplotlib_plot_WaveformSetGrid(
    wfset,
    detector=ordered_modules_membrane,
    plot_function=plot_averages,
    func_params={"mean_output": mean_output, "ppm_values": ppm_values},
    cols=2, 
    figsize=(14, 32)
    
)

In [ ]:
fig, axs = matplotlib_plot_WaveformSetGrid(
    wfset,
    detector=ordered_modules_cathode,
    plot_function=plot_averages,
    func_params={"mean_output": mean_output, "ppm_values": ppm_values},
    cols=2, 
    figsize=(14, 32)
    
)

In [ ]:
template = ch_read_template('templates_large_pulses')

In [ ]:
dfitparams = dict(
    scinttype = 'lar',
    error = 0.05,
    filter_type = 'Gauss',
    cutoff_MHz = 2.5,
    dtime = 16,
)
cfit = {}
for ep, ch_wvf in mean_output.items():
    for ch, _ in ch_wvf.items():
        if ep in template.keys():
            cfit[ep] = cfit.get(ep, {})
            if ch in template[ep].keys():
                cfit[ep][ch] = DeconvFitterVDWrapper(**dfitparams)

In [ ]:
def generate_deconv(ep:int, ch:int, response:np.ndarray, template:np.ndarray, cfitch:DeconvFitterVDWrapper, slice_template:slice = slice(200,240), slice_response:slice = slice(None,54)):
    setup_response_template(response, template, cfitch, slice_template, slice_response)
    if ep != 110:
        cfitch.generate_deconvolved_signal()
    else:
        cfitch.deconvolved = cfitch.response.copy()

In [ ]:
def plot_deconv(wfs, mean_output, ppm_values, average:bool = False, normalization:bool = False):
    ax = plt.gca()
    ch = wfs.channel
    ep = wfs.endpoint 

    prop_cycle_colors = [
    '#1f77b4',
    '#ff7f0e',
    '#2ca02c',
    '#d62728',
    '#9467bd',
    '#8c564b',
    '#e377c2',
    '#7f7f7f',
    '#bcbd22',
    '#17becf',]
    
    ppm_colors = {
        ppm: prop_cycle_colors[i % len(prop_cycle_colors)]
        for i, ppm in enumerate(sorted(ppm_values))
    }

    for ppm in sorted(ppm_values):
        if ppm not in mean_output:
            continue
        if ep not in mean_output[ppm]:
            continue
        if ch not in mean_output[ppm][ep]:
            continue
        if ep not in template or ch not in template[ep]:
            continue
            
        color = ppm_colors[ppm]
    
        if average:
            x_ns = np.arange(n_ticks) * 16
            if normalization:
                ax.plot(x_ns, mean_output[ppm][ep][ch]/np.max(mean_output[ppm][ep][ch]), label=f"{ppm} ppm", color=color)
            else:
                ax.plot(x_ns, mean_output[ppm][ep][ch], label=f"{ppm} ppm", color=color)
        else:
            wf = mean_output[ppm][ep][ch]
            cfitch = DeconvFitterVDWrapper(**dfitparams)
            generate_deconv(ep, ch, response=wf, template=template[ep][ch], cfitch=cfitch, slice_template=slice(200, 240), slice_response=slice(None, 54))
            
            n_ticks = len(cfitch.deconvolved)
            x_ns = np.arange(n_ticks) * 16

            if normalization:
                ax.plot(x_ns, cfitch.deconvolved/np.max(cfitch.deconvolved), label=f"{ppm} ppm", color=color)
            else:
                ax.plot(x_ns, cfitch.deconvolved, label=f"{ppm} ppm", color=color)            
            
            
    ax.set_title(f"{getModuleName(ep, ch)}: {ep}-{ch}")
    ax.set_xlabel("Time [ns]")
    ax.set_ylabel("Amplitude [a.u.]")
    # aaxsx.set_yscale('log')
    ax.legend(fontsize=12)

In [ ]:
fig, axs = matplotlib_plot_WaveformSetGrid(
    wfset,
    detector=ordered_modules_cathode,
    plot_function=plot_deconv,
    func_params={"mean_output": mean_output, "ppm_values": ppm_values, "average": False, "normalization": True},
    cols=2, 
    figsize=(14, 32)   
)
# plt.savefig("deconvolved_cathode_log_norm.png")

In [ ]:
fig, axs = matplotlib_plot_WaveformSetGrid(
    wfset,
    detector=["M4", "M5", "M6", "M7", "M8"],
    plot_function=plot_deconv,
    func_params={"mean_output": mean_output, "ppm_values": ppm_values, "average": False, "normalization": True },
    cols=2, 
    figsize=(14, 24)   
)
plt.savefig("deconvolved_membrane_log_norm.png")